# Causal Reasoning Deep Dive: Counterfactuals, Experiments, and Identification

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/honr_464_logo.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>HONR 46400 — Evidence-Driven Research</center>
# <center>Professor: Davi Moreira</center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/notebooks/student/nb13_causal_student.ipynb)

---

## 🧭 Approach & Claim Boundary

**Approach emphasis:** causal reasoning (deep dive) — the last and most demanding
column of the four-approach map. Prediction was content to forecast *who*;
causation insists on *because*, and the word must be earned. The coin of the
realm here is the **counterfactual** and the **identification argument** that
recovers it.

| | |
|---|---|
| **A claim this topic PERMITS** | "Because assignment was randomized (checked) — or nature randomized for me under a stated, probed assumption — the difference in outcomes carries a causal reading, with uncertainty." |
| **A claim this topic does NOT permit** | A causal claim whose counterfactual is *imagined* rather than *identified* ("they improved after the program, so the program worked"), or design vocabulary (DiD/RDD/IV) decorating a comparison whose assumptions were never argued. |

**Where this sits in the course:** meetings 27–28 of 44. It is the causal deep
dive that closes the four-approach arc, and it arms the causal branch of
milestone **M09 (Pilot Analysis)** — Friday is the pilot-walkthrough day.

*Provenance: RDSS ch. 18 'Experimental: causal' + declaration_18.1 (two-arm trial) + rdss foos_etal; RDSS ch. 16 'Observational: causal' + rdss cliningsmith_etal | experimental & observational causal designs | two-arm-trial concept translated; real experiments reproduced descriptively; DiD/RDD/IV as identification arguments (intuition only) | translated (R→Python) & adapted*

## Learning Objectives

By the end of this notebook, you will be able to:

1. State the **counterfactual** at the heart of any causal claim using potential
   outcomes Y(1) and Y(0), and name the **fundamental problem** that makes one of
   them always missing.
2. Explain how **randomization** manufactures a valid counterfactual, and write a
   three-sentence **identification argument**.
3. Analyze a **real randomized field experiment** — a difference in means, its
   uncertainty, and a balance check — and say exactly what you computed and what
   you did not.
4. Read **DiD, RDD, and IV** as identification *arguments* (intuition only), each
   with an assumption a skeptic would attack first.
5. Interpret a **natural experiment** (a lottery) and bound its claim honestly.
6. Write your **own project's identification paragraph** — or its honest absence —
   for the M09 pilot walkthrough.

---

In [ ]:
# Setup — run this cell first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (9, 5)

SEED = 464  # course number — keeps every simulation reproducible
rng = np.random.default_rng(SEED)

# Data loads: GitHub raw URL first (works in Colab), local repo path as fallback.
DATA_URL = ("https://raw.githubusercontent.com/davi-moreira/"
            "2026F_evidence_driven_research_purdue_HONR464/main/notebooks/data/")

def load_course_data(filename):
    """Load a course dataset from GitHub, falling back to the local repo copy."""
    try:
        return pd.read_csv(DATA_URL + filename)
    except Exception:
        from pathlib import Path
        local = Path("notebooks/data") / filename
        if not local.exists():
            local = Path("../data") / filename
        return pd.read_csv(local)

print("✓ Setup complete — seed", SEED)

## 1. Why This Matters — the missing world

> *"Someone always tells me the patients who took the drug got better. Fine — so
> did the ones who rested, who prayed, who were younger, who were richer. 'They
> got better after X' is not 'X worked'; it is a story missing its comparison.
> Show me what would have happened WITHOUT X, or admit you are guessing."*
> — a **skeptical peer** who refuses to confuse *after* with *because*

Every causal claim is secretly a claim about **two worlds**. For a single unit,
write Y(1) for the outcome if it receives the treatment and Y(0) for the outcome
if it does not. The causal effect *for that unit* is the difference,
Y(1) − Y(0): the world with the treatment minus the world without it.

Here is the catch that governs the entire field — the **fundamental problem of
causal inference**: for any one unit you can only ever observe *one* of those
worlds. A person either took the drug or did not; you never see both their
Y(1) and their Y(0). The counterfactual — the world that did not happen — is
missing by definition. No amount of staring at the data a person *did* generate
recovers the outcome they *would* have had under the other condition.

So how does anyone ever claim a cause? By giving up on the *individual* effect
and estimating the **average** effect across a group — and by engineering the
data so that one group can honestly stand in for another's missing world. That
engineering is **randomization**. Split many comparable units at random into a
treated arm and a control arm, and the two groups are alike *in expectation* on
everything — measured and unmeasured — so the control group's average outcome is
a valid estimate of what the treated group *would* have done untreated. The
missing world is borrowed from a comparable group.

An **identification argument** makes that borrowing explicit, in three sentences:
(1) the quantity I want is the average effect, Y(1) − Y(0), over my units; (2)
because treatment was assigned at random, the control group is a valid
counterfactual for the treated group; (3) therefore the difference in the two
groups' average outcomes estimates the causal effect, with uncertainty from
sampling.

Drawn as a diagram, a clean experiment is almost boringly simple — which is its
whole power:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/rdss_fig_18_1.png" width="60%"/></center>

*An experiment's causal diagram: a random assignment Z and a measurement procedure Q both feed the outcome Y, while unknown heterogeneity U also affects Y — but because Z is random, it is not tangled with U, which is exactly what lets the comparison carry a causal reading. (Figure from the replication materials of Blair, Coppock & Humphreys (2023), *Research Design in the Social Sciences* (MIT-licensed archive; the book is free at book.declaredesign.org).)*

> **A question that often comes up here:** *"If we can never see both outcomes
> for one person, isn't causal inference just impossible?"* The individual effect
> is genuinely unrecoverable — that part *is* impossible. What randomization
> rescues is the **average** effect across a group, because randomness makes the
> groups exchangeable. You trade the question "what did X do to *this* person?"
> for "what did X do *on average*?" — and the second question has an honest
> answer.

---

### 🔮 Pause & Predict

The `foos_etal` file is a **real** UK get-out-the-vote field experiment. Some
voters were assigned at random to a contact (treatment) arm; others were left as
a control arm; and whether each voter turned out in 2014 (`marked_register_2014`)
was recorded afterward. **Before running the next cell**, commit two predictions:
did being contacted *raise* turnout, and if so, by roughly how many percentage
points? Write a direction and a number.

### YOUR ANSWER HERE:

**Did contact raise turnout? (up / down / no change):**

**By roughly how many percentage points:**

**Why:**

---

### 🛠️ Run the Study — a real experiment, analyzed honestly

We will analyze the experiment with the **simplest honest tool**: the difference
in turnout between the treated arm and the control arm, plus its standard error
and a 95% interval. Two disciplines before the numbers appear:

- We say *exactly* what we compute — an **unweighted, simple difference in
  means** across the two arms.
- We say *exactly* what we do not. The study behind this replication dataset ran
  a more elaborate, **weighted** analysis (the `weights` column is the clue), so
  our number faithfully reproduces the experiment's *logic* — randomize, then
  compare arms — not the paper's headline estimate. Reproducing the logic is the
  lesson; overclaiming that we reproduced the paper would be the failure.

> 💡 **Gemini Prompt:** "This code analyzes a real get-out-the-vote experiment as a simple unweighted difference in turnout between the contacted arm and the control arm, with a standard error for two proportions and a 95% interval: [paste the next cell]. Explain why randomization is what lets a plain difference in means carry a causal reading, and predict whether the interval will exclude zero."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check the printed difference, its interval, and whether it excludes zero against Gemini's prediction — report the sign honestly whichever way it comes out.
> - [ ] Confirm this is the UNWEIGHTED difference: be ready to say why it reproduces the experiment's logic, not the paper's weighted headline estimate.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# Load the real experiment and confirm its shape before trusting anything.
foos = load_course_data("foos_etal.csv")
assert foos.shape == (8375, 5), "unexpected shape — flag this!"
print("✓ Loaded foos_etal.csv —", foos.shape[0], "rows,", foos.shape[1], "columns")
print()

# Group sizes and turnout by arm (UNWEIGHTED, simple).
arms = foos.groupby("treat")["marked_register_2014"].agg(n="count", turnout="mean")
arms.index = ["control (treat=0)", "contacted (treat=1)"]
print("Turnout by arm:")
print(arms.round(4))
print()

# Difference in means + its standard error (two proportions) + 95% interval.
t1 = foos.loc[foos.treat == 1, "marked_register_2014"]
t0 = foos.loc[foos.treat == 0, "marked_register_2014"]
diff = t1.mean() - t0.mean()
se = np.sqrt(t1.mean() * (1 - t1.mean()) / len(t1)
             + t0.mean() * (1 - t0.mean()) / len(t0))
ci_lo, ci_hi = diff - 1.96 * se, diff + 1.96 * se
print(f"difference (contacted - control): {diff:+.4f}  ({diff*100:+.1f} percentage points)")
print(f"standard error: {se:.4f}")
print(f"95% interval: [{ci_lo:+.4f}, {ci_hi:+.4f}]   (excludes 0: {ci_lo > 0})")

In [ ]:
# Self-check: the arms, the direction, and the interval are what we reported.
assert len(t1) == 6104 and len(t0) == 2271, "arm sizes changed — investigate before trusting"
assert diff > 0, "our estimate says contact RAISED turnout — report the sign honestly either way"
assert ci_lo > 0, "the 95% interval excludes zero"
print(f"✓ Self-check passed: contacted arm n={len(t1)}, control arm n={len(t0)}.")
print(f"✓ The unweighted difference in means is {diff*100:+.1f} points, "
      f"95% interval [{ci_lo*100:+.1f}, {ci_hi*100:+.1f}] points.")

### Balance — does the randomization look real?

Randomization's whole promise is that the arms are alike *before* treatment. That
is checkable: if the arms differ sharply on something fixed before contact, the
promise is in doubt. Run the cell below to compare the arms on a pre-treatment
trait (ward) and on the study's `weights`.

> 💡 **Gemini Prompt:** "This code compares the two experimental arms on a pre-treatment trait (ward) and on the study's design weights: [paste the next cell]. Explain what a balance check is supposed to confirm about randomization, and why arms that line up on wards but differ sharply on weights would signal that treatment was assigned with UNEQUAL probabilities across groups."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check the printed largest ward gap and the mean weight per arm — do the wards line up closely while the weights differ a lot, as Gemini describes?
> - [ ] Connect it to the estimate: be ready to say why differing weights make our simple unweighted difference a classroom approximation, not the paper's number.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# Balance check: are the arms similar on pre-treatment features?
print("Mean design weight by arm:")
print(foos.groupby("treat")["weights"].mean().round(3).rename(
    {0: "control", 1: "contacted"}).to_string())
print()
ward_share = pd.crosstab(foos["ward"], foos["treat"], normalize="columns")
ward_share.columns = ["control", "contacted"]
gap = (ward_share["contacted"] - ward_share["control"]).abs()
print(f"Largest gap in any ward's share between arms: {gap.max():.3f}")
print("(wards line up fairly closely; the weights, however, differ a lot by arm)")

### 🔍 Reading the Evidence — write the three-sentence identification argument

The balance check reveals something honest and important. The `weights` differ
*a lot* across arms (the control arm carries much heavier weights), which is a
fingerprint of a design that assigned treatment with **unequal probabilities**
across groups and corrects for it with weights. Our simple unweighted difference
ignores that correction — which is precisely why it is a classroom
approximation, not the paper's estimate. An honest researcher reports the method
they actually ran and flags what a fuller analysis would change.

In the cell below, write the **three-sentence identification argument** for this
experiment: (1) the quantity you want, (2) why the control arm is a valid
counterfactual for the treated arm, (3) what therefore follows — and add a fourth
sentence naming the one caveat our unweighted pass carries.

### YOUR ANSWER HERE:

**(1) The quantity I want:**

**(2) Why the control arm is a valid counterfactual:**

**(3) Therefore:**

**(4) The caveat my unweighted analysis carries:**

---

### 📝 Practice — name the counterfactual, then grade it

For each claim, write the **counterfactual it silently asserts** (the missing
world), then grade how plausibly that counterfactual is *identified* — low,
medium, or high — in one phrase.

- **A.** "People who attended the optional review session scored higher on the
  final, so the session raised scores."
- **B.** "The city cut its speed limit and crashes fell the next year, so the
  lower limit cut crashes."
- **C.** "In a coin-flip randomized trial, the group given the vaccine had fewer
  infections, so the vaccine reduced infections."


### YOUR ANSWER HERE:

**A — counterfactual / plausibility:**

**B — counterfactual / plausibility:**

**C — counterfactual / plausibility:**

---

### ⚖️ Make a Design Choice — identify, or stop at association

Now turn the lens on your own project. In the cell below, choose honestly between
two options and defend the choice in a short paragraph:

- **(a) I can identify a causal effect** — name the source of the counterfactual
  (a randomization you can run, or a natural experiment the world provides) and
  the assumption it rests on.
- **(b) I cannot identify a causal effect** — my honest claim stops at
  *association*, and I will word every finding that way ("X is associated with Y,"
  never "X causes Y").

Choosing (b) is not a defeat. A precisely-bounded associational claim is
publishable, honest research; a causal claim with no identification is a defect.

> 💡 **Gemini Prompt:** "I am arguing that [my
> treatment] causes [my outcome] in my project. Here is my identification
> argument and its key assumption: [paste it]. Act as a skeptical reviewer: list
> the strongest threats to my counterfactual — reasons my treated and comparison
> groups might not be comparable — and, for each, name the evidence that would
> rule it out."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check each threat the AI raises against how treatment was *actually*
>   assigned in your design — the AI cannot see your assignment mechanism.
> - [ ] Treat any "your design looks sound" reply as unearned reassurance, not a
>   verdict — you confirm an assumption with evidence, never with an AI's approval.
> - [ ] Log this use in your AI ledger: tool, task, verification.

### YOUR ANSWER HERE:

**My choice (a / b):**

**Defense (the counterfactual source and its assumption, OR why I stop at association):**

---

### 🎯 Project Transfer — your identification paragraph

Write the **identification paragraph** your project will carry (or its honest
absence). If your design identifies an effect, state the quantity, the
counterfactual, and why the counterfactual is valid. If it does not, write the
honest version: "My design does not identify a causal effect; the counterfactual
is not available, so my claim stops at association because ___." This paragraph
travels straight to Friday's M09 walkthrough and onto your poster.

### YOUR ANSWER HERE:

**My identification paragraph (or its honest absence):**

---

### 🎟️ Claim Ticket

**Claim Ticket #27** — your project's causal status in one sentence: *"My
project's central claim is identified by [randomization / natural experiment /
___] — or is honestly NOT identified, so it stops at association."*

### YOUR ANSWER HERE:

**My causal status, one sentence:**

---

---

*(Meeting M28 picks up here.)*

## 2. Causal Claims Without Experiments

**Guiding question:** *when I can't run an experiment, where might nature have run
one for me — and can I defend that it really did?*

> *"I don't get to randomize whether a country adopts a policy or whether a
> person lives through a war. Most questions I care about forbid the experiment.
> So my whole craft is finding the moments where the world randomized FOR me —
> and being brutally honest about whether it really did."*
> — the same **skeptical peer**, now working in the observational world

You usually cannot randomize. So the observational causal toolkit hunts for
places where *nature* supplies a credible counterfactual, and each tool is really
an **identification argument** with an assumption you must argue and probe — not a
formula you apply:

- **Difference-in-differences (DiD)** — two groups, one treated, both tracked over
  time. If, *absent* treatment, they would have moved in **parallel**, then the
  treated group's *extra* movement after treatment is the effect. The assumption
  to attack: parallel trends — would they really have moved together?
- **Regression discontinuity (RDD)** — treatment switches on at a sharp
  **threshold** of some score (a scholarship above a cutoff, a program below an
  income line). Units *just* below and *just* above the line are nearly
  identical except for treatment, so the **jump** in the outcome at the threshold
  is the effect. The assumption to attack: nothing else jumps at the cutoff, and
  units cannot precisely manipulate their score to land on the lucky side.
- **Instrumental variables (IV)** — a **nudge** that pushes some units toward
  treatment but reaches the outcome *only through* treatment. If the nudge is
  as-good-as-random and has no other path, it isolates a causal effect. The
  assumption to attack: is the nudge truly exogenous, with no back door to the
  outcome?

We will look at the first two as pictures — because the intuition lives in the
picture — then meet a real natural experiment.

> **A question that often comes up here:** *"If DiD, RDD, and IV can get causal
> answers without an experiment, why bother randomizing at all?"* Because each of
> these buys its causal reading with an assumption you must defend and a skeptic can
> attack — parallel trends, a clean unmanipulated cutoff, an instrument with no back
> door. Randomization makes the counterfactual valid *by construction*; the
> observational tools make it valid *by argument*, and arguments can be wrong. The
> experiment is the gold standard precisely because it needs no such assumption —
> which is why, when it is available, you take it.

---

### 🛠️ Hands-On: two illustrative worlds

The two cells below draw **simulated illustrations** (seeded, not real data) of
the worlds in which DiD and RDD identify an effect. They are teaching pictures:
each one is built so its assumption *holds*, so you can see what "it worked"
looks like — and therefore recognize what its absence would look like.

> 💡 **Gemini Prompt:** "This code simulates a difference-in-differences world where a treated and a control group would have moved in parallel absent treatment, then adds a known effect after treatment turns on: [paste the next cell]. Explain how DiD recovers the effect as the treated group's EXTRA movement beyond its parallel counterfactual, and predict that the printed post-treatment gap will equal the built-in effect."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check that the printed post-treatment gap really matches the built-in effect the comment names — the picture should recover exactly what was planted.
> - [ ] Name the assumption this clean illustration hides: it is BUILT to have parallel trends, so be ready to say what a violation would look like in real data.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# ILLUSTRATION 1 (simulated, not data): a parallel-trends / DiD world.
did_rng = np.random.default_rng(SEED)
time = np.array([0, 1, 2, 3])
post = time >= 2                       # treatment turns on between period 1 and 2
control = 40 + 3 * time + did_rng.normal(0, 0.3, 4)
effect = 6                             # the built-in causal effect
treated_cf = control + 8               # counterfactual: parallel to control, shifted up
treated = treated_cf + effect * post   # treated = counterfactual PLUS the effect, post-treatment

fig, ax = plt.subplots()
ax.plot(time, control, "o-", color="#4C72B0", label="control group")
ax.plot(time, treated, "s-", color="#C44E52", label="treated group (observed)")
ax.plot(time, treated_cf, "s--", color="#C44E52", alpha=0.45,
        label="treated counterfactual (parallel, no treatment)")
ax.axvline(1.5, color="gray", linestyle=":", label="treatment turns on")
ax.set_xlabel("time period")
ax.set_ylabel("outcome")
ax.set_title("ILLUSTRATION: difference-in-differences — "
             "the gap between the red lines after treatment IS the effect")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

post_gap = (treated - treated_cf)[post]
print(f"Built-in effect = {effect}. Post-treatment gap (treated - its counterfactual) = "
      f"{np.round(post_gap, 2).tolist()}  → the picture recovers the effect.")

> 💡 **Gemini Prompt:** "This code simulates a regression-discontinuity world where treatment switches on at a sharp cutoff of a running score, with a known jump at the threshold: [paste the next cell]. Explain why comparing units JUST below versus JUST above the cutoff identifies the effect, and why comparing points far apart on the running score would instead mix in the slope and mislead."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check the printed local jump (means just below vs just above the cutoff) against the built-in jump — does the local comparison recover it?
> - [ ] Name the assumption the illustration hides: nothing else changes at the cutoff and units cannot precisely manipulate their score — be ready to say why a real study must defend that.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# ILLUSTRATION 2 (simulated, not data): a threshold / RDD world.
rdd_rng = np.random.default_rng(SEED)
x = rdd_rng.uniform(0, 100, 300)       # a running score (e.g., an eligibility test)
cutoff = 50
jump = 8                               # the built-in effect: a jump at the cutoff
y = 20 + 0.3 * x + jump * (x >= cutoff) + rdd_rng.normal(0, 3, 300)

fig, ax = plt.subplots()
ax.scatter(x[x < cutoff], y[x < cutoff], s=14, color="#4C72B0", label="below cutoff (untreated)")
ax.scatter(x[x >= cutoff], y[x >= cutoff], s=14, color="#C44E52", label="at/above cutoff (treated)")
ax.axvline(cutoff, color="gray", linestyle=":", label=f"cutoff = {cutoff}")
ax.set_xlabel("running score")
ax.set_ylabel("outcome")
ax.set_title("ILLUSTRATION: regression discontinuity — "
             "the vertical CLIFF at the cutoff is the effect")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

# A LOCAL comparison near the line recovers the jump (the far-apart means would
# mix in the slope, so we compare only points just below vs just above).
lo = (x >= 44) & (x < 50)
hi = (x >= 50) & (x < 56)
print(f"Built-in jump = {jump}. Just below the cutoff, mean outcome = {y[lo].mean():.1f}; "
      f"just above, {y[hi].mean():.1f}; local jump ≈ {y[hi].mean() - y[lo].mean():.1f}.")

The same threshold logic, drawn as a causal diagram, is what "RDD" actually
claims:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/rdss_fig_16_8.png" width="60%"/></center>

*A regression-discontinuity diagram: a running variable X sets a cutoff-based treatment D, while X and unknown heterogeneity U both feed the outcome Y — so only right AT the cutoff, where X barely differs on either side, does the jump in D cleanly reveal its effect. (Figure from the replication materials of Blair, Coppock & Humphreys (2023), *Research Design in the Social Sciences* (MIT-licensed archive; the book is free at book.declaredesign.org).)*

### A real natural experiment — the Hajj lottery

Now leave the illustrations for real data. The `cliningsmith_etal` file is the
replication dataset behind the **Clingingsmith, Khwaja & Kremer** Hajj-lottery
study. Among people who *applied* to make the pilgrimage, a **lottery** decided
who received a visa (`success` = 1 for lottery winners). Because the lottery is
random *among applicants*, winners and losers are comparable — the world ran the
randomization for us. That is a **natural experiment**.

The `views_*` columns record attitudes toward other national and ethnic groups,
and `views` is a composite index of them. We compute the **simple mean
difference** in `views` between lottery winners and losers — again the
classroom-simple, unadjusted version, presented with its lottery-as-randomization
argument.

> 💡 **Gemini Prompt:** "This code computes the simple unadjusted mean difference in a composite 'views toward other groups' index between people who WON a visa lottery and those who lost, among applicants to the Hajj pilgrimage: [paste the next cell]. Explain why a lottery among applicants makes winners and losers comparable — a natural experiment — and why that limits the causal claim to applicants rather than to everyone."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check the printed difference and its interval — does it exclude zero, and does the sign match the direction you expected the pilgrimage to move views?
> - [ ] Confirm the claim's boundary: the lottery randomizes among applicants only, so be ready to say who this result does and does not speak for.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# Load the natural experiment and verify its columns/means before claiming anything.
hajj = load_course_data("cliningsmith_etal.csv")
assert hajj.shape == (958, 8), "unexpected shape — flag this!"
print("✓ Loaded cliningsmith_etal.csv —", hajj.shape[0], "rows,", hajj.shape[1], "columns")
print("Columns:", list(hajj.columns))
print()

views = hajj.groupby("success")["views"].agg(n="count", mean="mean")
views.index = ["lost lottery (success=0)", "won lottery (success=1)"]
print("Composite 'views toward other groups' by lottery outcome:")
print(views.round(4))
print()

w1 = hajj.loc[hajj.success == 1, "views"]
w0 = hajj.loc[hajj.success == 0, "views"]
d = w1.mean() - w0.mean()
se = np.sqrt(w1.var(ddof=1) / len(w1) + w0.var(ddof=1) / len(w0))
print(f"difference (won - lost): {d:+.4f}   SE: {se:.4f}   "
      f"95% interval: [{d - 1.96*se:+.4f}, {d + 1.96*se:+.4f}]")

In [ ]:
# Self-check: direction and interval match what we will claim.
assert hajj.shape == (958, 8)
assert w1.mean() > w0.mean(), "winners' mean should exceed losers' — report the sign honestly"
assert d - 1.96 * se > 0, "the 95% interval excludes zero"
print(f"✓ Self-check passed: lottery winners score {d:+.2f} higher on the composite "
      "views index, interval excluding zero.")
print("✓ This is a simple unadjusted difference — the classroom-simple version.")

### 🔍 Reading the Evidence — what the lottery lets you claim

In the cell below, answer two things about the roughly +0.47 difference. First:
what is the **most precise causal claim** the lottery permits — and over *whom*
(careful: the lottery randomizes among *applicants*, not among all people)?
Second: name two things this difference does **not** establish, no matter how
clean the lottery — for instance, the specific mechanism, or that our simple
unadjusted number equals the study's full estimate.

### YOUR ANSWER HERE:

**The most precise causal claim the lottery permits (and over whom):**

**Two things it does NOT establish:**

---

### Design-mimicry — the vocabulary is not the argument

One warning before you practice. The most common causal failure among earnest
researchers is **design mimicry**: borrowing the *vocabulary* — "difference-in-
differences," "regression discontinuity," "natural experiment" — to decorate a
comparison whose assumptions were never argued. Writing "DiD" does not make
parallel trends true; calling something a "natural experiment" does not make the
assignment random. The word *because* is earned by the **argued and probed
assumption**, never by the name of the method. A skeptic reads past the label
straight to the assumption — and so should you.

> **A question that often comes up here:** *"How do I know whether I have a real
> natural experiment or just a comparison I've labeled one?"* You interrogate the
> assignment, not the label. Ask the concrete question — was who got treated really
> as-good-as-random, and what specifically makes it so? — then state the one
> assumption a skeptic would attack and show the evidence that it holds. If the only
> thing making your comparison a "natural experiment" is that you called it one, you
> have the vocabulary and not the argument, and the word *because* has not been earned.

---

### 📝 Practice — match the design, then name the first attack

Match each scenario to its identification argument (DiD, RDD, or IV — each used
once), and write the **one sentence a skeptic would attack first** (the
assumption the whole claim rests on).

- **A.** "One state raised its minimum wage in 2015; a neighboring state did not.
  We compare each state's change in employment from 2014 to 2016."
- **B.** "A merit scholarship goes to applicants scoring at or above 1200 on a
  test; we compare college enrollment for applicants just below versus just above
  1200."
- **C.** "Rainfall on election day nudges some people to stay home; we use
  election-day rainfall to study how turnout affects which candidate wins."


### YOUR ANSWER HERE:

**A — design / first attack:**

**B — design / first attack:**

**C — design / first attack:**

---

### 🎯 Project Transfer — your pilot walkthrough

Today you present your **M09 pilot**. In the cell below, draft the three-beat
walkthrough you will deliver — *what I ran / what came out / what surprised me* —
and name the **one verification step** you most want a listener to push you on.
Bring the identification paragraph you wrote on Wednesday: if your pilot is
causal, its honesty lives there; if it is descriptive or predictive, state the
claim boundary your walkthrough will respect ("association, not cause";
"held-out, not explanation"). A pilot that *collapsed* during the week is welcome
— present the failure and your diagnosis; that is legitimate, gradable research.

### YOUR ANSWER HERE:

**What I ran:**

**What came out:**

**What surprised me:**

**The one verification step I want pushed on:**

---

### 🎟️ Claim Ticket

**Claim Ticket #28** — the verification question a listener asked you today, and
how you will answer it before the poster.

### YOUR ANSWER HERE:

**The verification question + my plan to answer it:**

---

## 3. Wrap-Up

Across two meetings you learned what it takes to earn the word *because*. You met
the **counterfactual** — the missing world Y(0) or Y(1) you never get to see for
one unit — and watched **randomization** rescue the *average* effect by making
one group a valid stand-in for another's missing world. You analyzed a **real**
get-out-the-vote experiment honestly: a +3.4-point unweighted difference with an
interval excluding zero, flagged openly as the classroom-simple version of a
weighted study. Then you left the lab for the world's own experiments — the
parallel lines of DiD, the cliff of RDD, the nudge of IV, and the Hajj lottery's
+0.47-point natural experiment — each an *argument* whose one assumption a
skeptic attacks first.

> **"The word 'because' is not earned by a method's name — it is earned by a
> counterfactual you can defend."**

On Monday the course turns from finding evidence to **communicating** it: nb14
opens the poster phase. Everything you have built — a claim, an approach, and an
honest boundary — now has to survive a stranger's ninety-second read. Bring your
best figure and your headline claim.

---

## 4. Sources & Provenance

**Provenance of this notebook:**
- *RDSS ch. 18 'Experimental: causal' + declaration_18.1 (two-arm trial) | experimental causal designs | two-arm-trial concept; real GOTV experiment reproduced as an unweighted difference in means | translated (R→Python) & adapted*
- *RDSS ch. 16 'Observational: causal' | observational causal designs | DiD/RDD/IV as identification arguments, intuition only, no estimation machinery | adapted*
- *fresh | two seeded illustrative simulations (parallel-trends world, threshold world) | — | fresh*
- *foos_etal.csv | rdss package data | Foos et al. UK get-out-the-vote field experiment replication; unweighted classroom analysis (the study's own analysis is weighted) | adapted*
- *cliningsmith_etal.csv | rdss package data | Clingingsmith, Khwaja & Kremer Hajj-lottery study replication; simple unadjusted mean difference | adapted*

**Dataset attribution:** Datasets from the `rdss` package (Blair, Coppock &
Humphreys, MIT License), companion to *Research Design in the Social Sciences*
(2023).

**Readings:**
- Blair, G., Coppock, A., & Humphreys, M. (2023). *Research Design in the Social
  Sciences*, ch. 16 'Observational: causal' and ch. 18 'Experimental: causal'.
  Free online: [book.declaredesign.org](https://book.declaredesign.org/).

---

<center>

Thank you!

</center>